In [ ]:
from pathlib import Path
from io import StringIO
import re
import pandas as pd
from bs4 import BeautifulSoup

running_results = []
field_results = []

for year in range(2022, 2027):
    for gender in ["Men", "Women"]:
        filename = f"bigwest_{year}_{gender.lower()}.html"
        path = Path("../data/raw") / filename

        field_tables = FIELD_TABLES[(year, gender)]

        running, field = process_championship(
            path, gender, year, field_tables
        )

        running_results.append(running)
        field_results.append(field)

running_df = pd.concat(running_results, ignore_index=True)
field_df = pd.concat(field_results, ignore_index=True)


NameError: name 'FIELD_TABLES' is not defined

: 

In [ ]:
FIELD_KEYWORDS = ('long jump', 'triple jump', 'high jump', 'pole vault',
                  'shot put', 'discus', 'hammer', 'javelin')
COMBINED_KEYWORDS = ('heptathlon', 'decathlon', 'pentathlon')

def get_event_info(table_tag):
    heading = table_tag.find_previous(['h1', 'h2', 'h3', 'h4', 'h5'])
    while heading:
        text = heading.get_text(' ', strip=True)
        if re.search(r"(?:Women['’]s|Men['’]s)", text, re.I):
            break
        heading = heading.find_previous(['h1', 'h2', 'h3', 'h4', 'h5'])
    if not heading:
        return None, None, None, None
    text = heading.get_text(' ', strip=True)
    event = re.split(r'\b(?:Finals?|Preliminaries|Semi-Finals?|Flight|Heat|Section)\b',
                     text, maxsplit=1, flags=re.I)[0]
    event = re.sub(r"^(?:Women['’]s|Men['’]s)\s+", '', event).strip()
    low = event.lower()
    event_type = ('combined' if any(x in low for x in COMBINED_KEYWORDS)
                  else 'field' if any(x in low for x in FIELD_KEYWORDS)
                  else 'running')
    round_name = ('Flight' if re.search(r'\bFlight\b', text, re.I)
                  else 'Heat' if re.search(r'\bHeat\b', text, re.I)
                  else 'Preliminaries' if re.search(r'\bPreliminaries\b', text, re.I)
                  else 'Finals' if re.search(r'\bFinals?\b', text, re.I)
                  else 'Other')
    return event, event_type, round_name, text

# Main overall results tables for the 2026 women's saved HTML.
# Indices are specific to this file; check them when changing years/genders.
WOMEN_FIELD_TABLES = {
    'High Jump': 48, 'Pole Vault': 49, 'Long Jump': 50,
    'Triple Jump': 53, 'Shot Put': 56, 'Discus': 59,
    'Hammer': 62, 'Javelin': 65,
}

# Only Triple Jump MARK.2 has been independently checked against the
# previously verified results. Other multi-MARK selections need review.
WOMEN_VERIFIED_MARK_COLUMNS = {'Triple Jump': 'MARK.2', 'Discus': 'MARK', 'Hammer': 'MARK'}

def is_field_summary_table(df):
    return {'PL', 'NAME', 'TEAM', 'MARK'}.issubset(df.columns)

def parse_time_seconds(value):
    """Convert sprint or mm:ss.xx race times into seconds."""
    if pd.isna(value):
        return float('nan')
    try:
        pieces = str(value).strip().split(':')
        return sum(float(piece) * 60**power
                   for power, piece in enumerate(reversed(pieces)))
    except (ValueError, TypeError):
        return float('nan')


def choose_running_time_column(df):
    """Find the TIME column that follows official finish-place ordering.

    Saved HTML may contain multiple TIME columns with spurious values.
    The real results should be nondecreasing by finish place. If the
    ordering cannot identify a unique column, fail rather than guess.
    """
    cols = [c for c in df.columns if re.fullmatch(r'TIME(?:\.\d+)?', str(c))]
    if not cols:
        return None
    if len(cols) == 1:
        return cols[0]
    places = pd.to_numeric(df['PL'], errors='coerce')
    scores = []
    for col in cols:
        seconds = df[col].map(parse_time_seconds)
        valid = places.notna() & seconds.notna()
        if valid.sum() < 3:
            continue
        ordered = pd.DataFrame({'place': places[valid], 'time': seconds[valid]}).sort_values('place')
        # Equal places can represent ties; ignore their relative order.
        diffs = ordered['time'].diff()
        place_diffs = ordered['place'].diff()
        violations = int(((diffs < -0.001) & (place_diffs > 0)).sum())
        scores.append((violations, col))
    if not scores:
        return None
    best = min(s[0] for s in scores)
    winners = [col for violations, col in scores if violations == best]
    if best != 0 or len(winners) != 1:
        return None
    return winners[0]


def get_official_mark_column(df, event_type, round_name, event=None, verified_marks=None):
    if event_type == 'field':
        cols = [c for c in df.columns if re.fullmatch(r'MARK(?:\.\d+)?', str(c))]
        if not cols:
            return None
        return (verified_marks or {}).get(event, cols[-1])
    return choose_running_time_column(df)

# Men's main field tables are discovered by event heading, not by women's table numbers.
def find_main_field_tables(tables, table_tags):
    found = {}
    for i, (table, tag) in enumerate(zip(tables, table_tags)):
        event, event_type, round_name, _ = get_event_info(tag)
        if (event_type == 'field' and round_name not in ('Flight', 'Heat', 'Preliminaries')
                and is_field_summary_table(table)):
            if event in found:
                raise ValueError(f'Multiple main field tables for {event}: {found[event]}, {i}')
            found[event] = i
    return found


In [ ]:
# Women's main field table numbers are known from the previously inspected HTML.
print('Women main field tables:', WOMEN_FIELD_TABLES)
print('Men main field tables will be detected from the saved men HTML.')


In [ ]:
def clean_event_table(table, tag, index, gender, year, field_tables, verified_marks):
    event, event_type, round_name, heading = get_event_info(tag)
    if not event or event_type == 'combined':
        return None
    df = table.copy()
    if event_type == 'field':
        # Use only the main overall results table, not the flight tables.
        if index != field_tables.get(event):
            return None
        if not is_field_summary_table(df):
            return None
    elif not {'PL', 'NAME', 'TEAM'}.issubset(df.columns):
        return None

    mark_col = get_official_mark_column(df, event_type, round_name, event, verified_marks)
    if mark_col is None:
        if event_type == 'running':
            print(f'WARNING: ambiguous running TIME columns: {gender} table {index} {heading!r}; skipping')
        return None
    df = df.rename(columns={'PL':'place', 'NAME':'athlete_name', 'YEAR':'class_year',
                            'TEAM':'school', 'SC':'points', 'WIND':'wind',
                            'CONV':'converted_mark', mark_col:'mark'})
    ids = []
    for tr in tag.select('tbody tr'):
        link = tr.select_one('a[href*="/athletes/"]')
        m = re.search(r'/athletes/(\d+)', link.get('href', '')) if link else None
        ids.append(m.group(1) if m else None)
    if len(ids) != len(df):
        print(f'SKIP table {index}: athlete-link rows {len(ids)} != parsed rows {len(df)}')
        return None
    df['athlete_id'] = ids
    df = df[df['athlete_id'].notna() & df['athlete_name'].notna() & df['school'].notna()].copy()
    if df.empty:
        return None
    for optional in ('class_year', 'wind', 'converted_mark'):
        if optional not in df: df[optional] = None
    if 'points' not in df: df['points'] = 0
    df['points'] = pd.to_numeric(df['points'], errors='coerce').fillna(0)
    df['place'] = pd.to_numeric(df['place'], errors='coerce')
    df['year'], df['gender'], df['event'] = year, gender, event
    df['event_type'], df['round'] = event_type, round_name
    df['source_table'], df['source_heading'] = index, heading
    df['mark_source_column'] = mark_col
    return df[['year','gender','event','event_type','round','athlete_id',
               'athlete_name','class_year','school','place','mark','converted_mark',
               'points','wind','source_table','source_heading','mark_source_column']]


def process_championship(path, gender, year, field_tables=None, verified_marks=None):
    if not path.exists():
        raise FileNotFoundError(f'Missing saved {gender} HTML: {path}')
    soup = BeautifulSoup(path.read_text(encoding='utf-8'), 'html.parser')
    # Validate actual results headings, not the file name.
    event_headings = [h.get_text(' ', strip=True) for h in soup.find_all(['h1','h2','h3','h4','h5'])]
    # Anchor to the beginning: "Men's" is a substring of "Women's"!
    expected = r"^Women['’]s\b" if gender == 'Women' else r"^Men['’]s\b"
    opposite = r"^Men['’]s\b" if gender == 'Women' else r"^Women['’]s\b"
    matching = [h for h in event_headings if re.search(expected, h, re.I)]
    conflicting = [h for h in event_headings if re.search(opposite, h, re.I)]
    if not matching or conflicting:
        raise ValueError(
            f"Wrong results HTML for {gender}: {path}. "
            f"Matching headings={len(matching)}, opposite-gender headings={len(conflicting)}. "
            "Save the correct results page and rerun all cells."
        )
    for tag in soup.find_all(attrs={'colspan': True}):
        if not str(tag['colspan']).isdigit():
            tag['colspan'] = '1'
    table_tags = soup.find_all('table')
    tables = pd.read_html(StringIO(str(soup)))
    if len(table_tags) != len(tables):
        raise ValueError('HTML tables and pandas tables are not aligned')
    if field_tables is None:
        field_tables = find_main_field_tables(tables, table_tags)
    print(f'{gender} field tables:', field_tables)
    expected_fields = set(WOMEN_FIELD_TABLES)
    missing_fields = expected_fields - set(field_tables)
    if missing_fields:
        print(f'WARNING: {gender} missing main field tables:', sorted(missing_fields))
    parsed = []
    for i, (table, tag) in enumerate(zip(tables, table_tags)):
        result = clean_event_table(table, tag, i, gender, field_tables, verified_marks or {})
        if result is not None:
            parsed.append(result)
    if not parsed:
        raise ValueError(f'No athlete results extracted from {path}')
    results = pd.concat(parsed, ignore_index=True)
    priority = {'Finals': 4, 'Other': 3, 'Preliminaries': 2, 'Flight': 1, 'Heat': 1}
    results['round_priority'] = results['round'].map(priority).fillna(0)
    results['has_mark'] = results['mark'].notna() & results['mark'].astype(str).str.strip().ne('')
    results = results.sort_values(['round_priority', 'has_mark', 'points', 'source_table'],
                                  ascending=[False, False, False, True])
    results = results.drop_duplicates(subset=['year','gender','event','athlete_id'], keep='first')
    field = results[results['event_type'] == 'field'].copy()
    running = results[results['event_type'] == 'running'].copy()
    assert field.apply(lambda r: int(r['source_table']) == field_tables.get(r['event']), axis=1).all()
    return running, field

women_running, women_field = process_championship(
    women_source, 'Women', WOMEN_FIELD_TABLES, WOMEN_VERIFIED_MARK_COLUMNS)
men_running, men_field = process_championship(
    men_source, 'Men')  # Men mark columns are provisional and require verification.

# Fail rather than export identical athlete/event results with opposite gender labels.
for kind, womens, mens in [('running', women_running, men_running), ('field', women_field, men_field)]:
    left = set(zip(womens['event'], womens['athlete_id']))
    right = set(zip(mens['event'], mens['athlete_id']))
    if left and right and left == right:
        raise ValueError(f'{kind}: Men and Women have identical event/athlete IDs. Check input HTML files.')

running_df = pd.concat([women_running, men_running], ignore_index=True)
field_df = pd.concat([women_field, men_field], ignore_index=True)
print('Combined running rows:', len(running_df))
print('Combined field rows:', len(field_df))


In [ ]:
# Both genders are in the same running and field dataframes.
assert set(running_df['gender']) == {'Women', 'Men'}
assert set(field_df['gender']) == {'Women', 'Men'}
print('RUNNING BY GENDER AND EVENT')
display(running_df.groupby(['gender', 'event']).size().rename('athletes'))
print('FIELD BY GENDER AND EVENT')
display(field_df.groupby(['gender', 'event']).agg(
    athletes=('athlete_id', 'nunique'), points=('points', 'sum'),
    source_table=('source_table', 'first'), mark_column=('mark_source_column', 'first')))


In [ ]:
# Audit running TIME column selection and flag events missing official times.
print('RUNNING TIME AUDIT')
display(running_df.groupby(['gender', 'event', 'mark_source_column']).agg(
    athletes=('athlete_id', 'nunique'),
    example_mark=('mark', 'first')
).reset_index())
print('NOTE: Multiple TIME columns are resolved only when exactly one follows finish-place order.')

In [ ]:
print('FIELD MARK AUDIT — verify the official mark column and values before modeling.')
for gender in ('Women', 'Men'):
    for event in ('High Jump', 'Pole Vault', 'Long Jump', 'Triple Jump', 'Shot Put', 'Discus', 'Hammer', 'Javelin'):
        sub = field_df[(field_df['gender'] == gender) & (field_df['event'] == event)]
        if sub.empty:
            print('MISSING:', gender, event)
        else:
            print(gender, event, '| mark column:', sub['mark_source_column'].iloc[0])
            display(sub[['athlete_name','place','mark','points']].sort_values('place').head(3))
print('NOTE: Unverified field MARK selections and running TIME selections are provisional.')


In [ ]:
# Export two combined datasets, not separate files per gender.
running_df.to_csv(
    "../data/processed/bigwest_2022_2026_running.csv",
    index=False
)

field_df.to_csv(
    "../data/processed/bigwest_2022_2026_field.csv",
    index=False
)
